# Question answering with a pretrained CamemBERT

Runs extractive QA with [`fmikaelian/camembert-base-fquad`](https://huggingface.co/fmikaelian/camembert-base-fquad) and plots the start/end scores the model assigns to each token.

In [ ]:
import torch
from transformers.modeling_camembert import CamembertForQuestionAnswering
from transformers.tokenization_camembert import CamembertTokenizer

In [ ]:
model_path = 'fmikaelian/camembert-base-fquad'
tokenizer = CamembertTokenizer.from_pretrained(model_path, do_lower_case=True)
model = CamembertForQuestionAnswering.from_pretrained(model_path)

In [ ]:
def eval_text(question,text):
    input_ids = tokenizer.encode(question, text)
    start_scores, end_scores = model(torch.tensor([input_ids]))
    all_tokens = tokenizer.convert_ids_to_tokens(input_ids)
    start = torch.argmax(start_scores)
    end = torch.argmax(end_scores)+1
    return ''.join(all_tokens[start:end]).replace('▁',' ').strip()

In [ ]:
question = "Où est fondée la nouvelle capitale ?"
context ="Après la prise de Babylone par les Hittites, la situation politique de la Babylonie est particulièrement obscure. Cette région tombe dans des conditions mal établies sous la coupe d'une dynastie d'origine kassite. Un texte du VIIe siècle av. J.-C. retrouvé à Ninive en Assyrie se présente comme une copie d'une inscription du roi kassite Agum II, qui dit avoir rapporté les statues de culte de Marduk et Sarpanitu à Babylone et restauré l'Esagil. On ne sait rien quant à l'authenticité de ce texte, d'autant plus que ce roi Agum n'est mentionné que dans des textes postérieurs à son règne. En l'état actuel des connaissances la domination kassite sur la Babylonie n'est assurée que vers le début du XVe siècle av. J.-C., sous Burna-Buriash Ier et surtout ses successeurs Ulam-Buriash et Agum III. Les rois de cette dynastie, qui se présentent comme monarques de « Karduniaš » (région correspondant à la Babylonie) plus souvent que comme « rois de Babylone », n'apparaissent que rarement en relation avec la ville de Babylone où leurs travaux ne sont pas ou peu mentionnés. Le statut de Babylone en tant que centre politique n'est pas clair : sous Kurigalzu Ier (ou le deuxième du nom) au début du XIVe siècle av. J.-C., une nouvelle capitale est fondée à Dûr-Kurigalzu (« fort Kurigalzu », du nom de son fondateur), située plus au nord dans la région où l'Euphrate et le Tigre sont proches, comme d'autres futures capitales des royaumes établis en Mésopotamie (Séleucie du Tigre, Ctésiphon puis Bagdad)."
eval_text(question,context)

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use plot styling from seaborn.
sns.set(style='darkgrid')
# Increase the plot size and font size.
plt.rcParams["figure.figsize"] = (16,8)

def visualize(tokens, start_scores, end_scores):
    # Pull the scores out of PyTorch Tensors and convert them to 1D numpy arrays.
    s_scores = start_scores.detach().numpy().flatten()
    e_scores = end_scores.detach().numpy().flatten()

    # We'll use the tokens as the x-axis labels. In order to do that, they all need
    # to be unique, so we'll add the token index to the end of each one.
    token_labels = []
    for (i, token) in enumerate(tokens):
        token_labels.append('{:} - {:>2}'.format(token, i))
    
    # Create a barplot showing the start word score for all of the tokens.
    ax = sns.barplot(x=token_labels, y=s_scores, ci=None)

    # Turn the xlabels vertical.
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="center")

    # Turn on the vertical grid to help align words to scores.
    ax.grid(True)

    plt.title('Start Word Scores')

    plt.show()
    # Create a barplot showing the end word score for all of the tokens.
    ax = sns.barplot(x=token_labels, y=e_scores, ci=None)

    # Turn the xlabels vertical.
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="center")

    # Turn on the vertical grid to help align words to scores.
    ax.grid(True)

    plt.title('End Word Scores')

    plt.show()

In [ ]:
input_ids = tokenizer.encode(question, context)
tokens = tokenizer.convert_ids_to_tokens(input_ids)
start_scores, end_scores = model(torch.tensor([input_ids]))

visualize(tokens,start_scores,end_scores)